# 02 Collect results
Gathers the per-image scores and run summaries of all finished runs into `BUSI_QB/analysis/` (read only for `runs/`). The statistics of protocol Section 7 are computed from these files.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, json, time, shutil
import numpy as np, pandas as pd
from concurrent.futures import ThreadPoolExecutor
ROOT = '/content/drive/MyDrive/BUSI_QB'
RUNS = os.path.join(ROOT, 'runs'); PREP = os.path.join(ROOT, 'prepared')
OUT = os.path.join(ROOT, 'analysis'); os.makedirs(OUT, exist_ok=True)
print(sorted(os.listdir(ROOT)))

In [ ]:
# COLLECT: reads every finished run of plans SEL, A, B, C and writes a few compact files to BUSI_QB/analysis.
# Read only for runs/: nothing in runs/ is changed. No image is read or written.
t0 = time.time()
dirs = sorted(d for p in ('SEL', 'A', 'B', 'C') for d in glob.glob(os.path.join(RUNS, p, '*')) if os.path.isdir(d))
def load(d):
    s = os.path.join(d, 'summary.json')
    if not os.path.exists(s):
        return None
    out = dict(summary=json.load(open(s)))
    for nm in ('test_busi', 'test_busbra', 'test_breast', 'normals_busi', 'code_replacement', 'finite_shot'):
        f = os.path.join(d, nm + '.csv')
        if os.path.exists(f):
            t = pd.read_csv(f); t.insert(0, 'run_id', out['summary']['run_id']); out[nm] = t
    return out
with ThreadPoolExecutor(16) as ex:
    res = [r for r in ex.map(load, dirs) if r is not None]
print(len(res), 'runs read in', round((time.time() - t0) / 60, 1), 'min')
summ = pd.DataFrame([{k: (json.dumps(v) if isinstance(v, (dict, list)) else v) for k, v in r['summary'].items()} for r in res])
summ.to_csv(os.path.join(OUT, 'summaries.csv'), index=False)
print(summ.groupby('plan').size().to_dict())
for nm in ('test_busi', 'test_busbra', 'test_breast', 'normals_busi', 'code_replacement', 'finite_shot'):
    parts = [r[nm] for r in res if nm in r]
    if parts:
        t = pd.concat(parts, ignore_index=True)
        for c in t.columns:
            if t[c].dtype == float:
                t[c] = t[c].round(5)
        t.to_csv(os.path.join(OUT, nm + '.csv.gz'), index=False, compression='gzip')
        print(nm, len(t), 'rows', round(os.path.getsize(os.path.join(OUT, nm + '.csv.gz')) / 1e6, 1), 'MB')
for f in ('busi_manifest.csv',):
    shutil.copy(os.path.join(PREP, f), os.path.join(OUT, f))
shutil.copy(os.path.join(RUNS, 'plan_decision.json'), os.path.join(OUT, 'plan_decision.json'))
print('COLLECT OK', sorted(os.listdir(OUT)), round((time.time() - t0) / 60, 1), 'min')

In [ ]:
# SPLIT: BUS-BRA scores in four smaller files (each under 10 MB), so they can be downloaded one by one.
t = pd.read_csv(os.path.join(OUT, 'test_busbra.csv.gz'))
t['part'] = t.run_id.str.extract(r'^(A|C)_')[0] + t.run_id.str.extract(r'(_p\d{3})$')[0].fillna('')
keep = ['run_id', 'image', 'patient', 'dice', 'iou', 'hd95', 'empty']
for part, g in t.groupby('part'):
    f = os.path.join(OUT, f'test_busbra_{part}.csv.gz')
    g[keep].to_csv(f, index=False, compression='gzip')
    print(part, len(g), 'rows', round(os.path.getsize(f) / 1e6, 1), 'MB')
print('SPLIT OK')